# ♻️ AI Smart Recycle Bin - Waste Classification Training in Google Colab

This notebook trains and evaluates a **TensorFlow / Keras Deep Learning Model** for classifying waste materials (**Metal, Paper, Plastic**) on **Google Colab** and exports a quantized **TensorFlow Lite (`best.tflite`)** model for deployment on the **Raspberry Pi 4 / 5**.

### Workflow:
1. **Environment Setup & GPU Verification**
2. **Dataset Acquisition & Train/Test Split Generation**
3. **Data Augmentation & Image Preprocessing (`tf.keras`)**
4. **Model Architecture & Transfer Learning (MobileNetV2 / CNN)**
5. **Training the Model on Google Colab**
6. **Testing & Performance Evaluation (Accuracy, Loss Curves, Confusion Matrix)**
7. **TensorFlow Lite Conversion (`best.tflite`) & Download**

## 1. Verify TensorFlow and GPU Setup

In [ ]:
import os
import shutil
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
from google.colab import files

print(f"TensorFlow Version: {tf.__version__}")
print("GPU Devices Available:", tf.config.list_physical_devices('GPU'))

## 2. Download Dataset & Generate Train / Test Splits
We use the Garbage Classification dataset containing **Metal**, **Paper**, and **Plastic** samples, and partition it into **Training (80%)** and **Testing (20%)** sets.

In [ ]:
# Download the waste classification dataset
!wget -q -O dataset.zip "https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip"

with zipfile.ZipFile("dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("dataset_raw")

# Create structured Train and Test directories
base_dir = "dataset_raw/dataset-resized"
train_dir = "dataset/train"
test_dir = "dataset/test"

target_classes = ["metal", "paper", "plastic"]
split_ratio = 0.8  # 80% Train, 20% Test

for cls in target_classes:
    os.makedirs(os.path.join(train_dir, cls), exist_ok=True)
    os.makedirs(os.path.join(test_dir, cls), exist_ok=True)
    
    src_cls_dir = os.path.join(base_dir, cls)
    if os.path.exists(src_cls_dir):
        images = [f for f in os.listdir(src_cls_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]
        np.random.shuffle(images)
        
        split_idx = int(len(images) * split_ratio)
        train_imgs = images[:split_idx]
        test_imgs = images[split_idx:]
        
        for img in train_imgs:
            shutil.copy(os.path.join(src_cls_dir, img), os.path.join(train_dir, cls, img))
        for img in test_imgs:
            shutil.copy(os.path.join(src_cls_dir, img), os.path.join(test_dir, cls, img))
            
        print(f"Class '{cls}': {len(train_imgs)} Train images, {len(test_imgs)} Test images")

## 3. Data Preprocessing & Augmentation with TensorFlow

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Data augmentation for training to prevent overfitting
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Test generator (rescaling only)
test_datagen = ImageDataGenerator(rescale=1.0 / 255.0)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("Class Mapping:", train_generator.class_indices)

## 4. Build TensorFlow Model Architecture (MobileNetV2 Transfer Learning)
MobileNetV2 provides high accuracy with lightweight computational overhead, ideal for edge deployment on Raspberry Pi.

In [ ]:
# Load pre-trained MobileNetV2 base
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze base weights during initial training

# Classification head
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(len(target_classes), activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 5. Train the Model on Google Colab

In [ ]:
EPOCHS = 20

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=test_generator,
    callbacks=callbacks
)

## 6. Test Model & Plot Performance Metrics

In [ ]:
# Plot Accuracy and Loss curves
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy', color='blue')
plt.plot(history.history['val_accuracy'], label='Test/Val Accuracy', color='orange')
plt.title('Training and Testing Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss', color='blue')
plt.plot(history.history['val_loss'], label='Test/Val Loss', color='orange')
plt.title('Training and Testing Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# Evaluate on Test Dataset
test_loss, test_acc = model.evaluate(test_generator)
print(f"\nFinal Test Accuracy: {test_acc * 100:.2f}%")
print(f"Final Test Loss: {test_loss:.4f}")

# Confusion Matrix & Classification Report
test_generator.reset()
predictions = model.predict(test_generator)
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes

print("\n--- Classification Report ---")
print(classification_report(y_true, y_pred, target_names=target_classes))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_classes, yticklabels=target_classes)
plt.title('Test Dataset Confusion Matrix')
plt.xlabel('Predicted Class')
plt.ylabel('True Class')
plt.show()

## 7. Convert to TensorFlow Lite (`best.tflite`) for Raspberry Pi

In [ ]:
# Convert Keras model to TensorFlow Lite format with dynamic range quantization
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

# Save model file
tflite_path = "best.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print(f"TensorFlow Lite model saved successfully: '{tflite_path}'")
print(f"Model Size: {os.path.getsize(tflite_path) / (1024 * 1024):.2f} MB")

# Download model file to deploy onto Raspberry Pi
files.download(tflite_path)